### ЗАДАЧА: Бронирование переговорок

Офис-менеджер получает пачку заявок на переговорки.
Нужно собрать систему, которая:
- принимает корректные бронирования,
- отбрасывает конфликтующие или неправильные заявки,
- хранит расписание по комнатам,
- позволяет быстро понять, какая переговорка загружена сильнее всего.

В некоторых заявках указана неизвестная комната,
в некоторых время перепутано,
а некоторые пересекаются с уже занятыми слотами.


In [20]:
from dataclasses import dataclass
from typing import Optional


rooms = {'A-101', 'B-204', 'C-305'}
# rows: booking_id|room_id|owner|start_hour|end_hour
rows = [
    'BK-100|A-101|Alice|9|11',
    'BK-101|A-101|Bob|10|12',
    'BK-102|B-204|Kira|13|15',
    'BK-103|X-999|Oleg|11|12',
    'BK-104|C-305|Eva|16|15',
    'BK-105|B-204|Max|15|17',
]


class BookingError(Exception):
    pass


class RoomNotFoundError(BookingError):
    pass


class TimeRangeError(BookingError):
    pass


class TimeConflictError(BookingError):
    pass


@dataclass(order=True)
class Booking:
    start_hour: int
    end_hour: int
    booking_id: str
    room_id: str
    owner: str


class RoomSchedule:
    def __init__(self, room_id):
        # TODO: сохранить room_id
        self.room_id = room_id
        
        # TODO: создать список bookings
        self.bookings = []
        

    def can_add(self, booking):
        for existing in self.bookings:
              if not (booking.end_hour <= existing.start_hour or booking.start_hour >= existing.end_hour):
                return False
        return True
        # TODO: пройтись по уже существующим booking в self.bookings
        # TODO: проверить пересечение интервалов
        # TODO: если пересечение есть -> вернуть False
        # TODO: если конфликтов нет -> вернуть True
        

    def add_booking(self, booking):
        if not self.can_add(booking):
            raise TimeConflictError()
            self.bookings.append(booking)
            self.bookings.sort()
        # TODO: если can_add(...) == False -> raise TimeConflictError(...)
        # TODO: добавить booking в self.bookings
        # TODO: отсортировать self.bookings
        

    def total_reserved_hours(self):
        return sum(b.end_hour - b.start_hour for b in self.bookings)
        # TODO: вернуть сумму (end_hour - start_hour) по всем бронированиям комнаты
       


class BookingService:
    def __init__(self, rooms):
        self.schedules = {room_id: RoomSchedule(room_id) for room_id in rooms}
        self.accepted = []
        self.errors = []
        # TODO: создать schedules вида room_id -> RoomSchedule(room_id)
        # TODO: создать списки accepted и errors
        

    def parse_booking(self, row):
        parts = row.split("|")
        if len(parts) != 5:
            raise BookingError()
        booking_id, room_id, owner, start_raw, end_raw = parts
        try:
            start_hour = int(start_raw)
            end_hour = int(end_raw)
        except ValueError as e:
            raise BookingError(f"не правильное значение времени: start={start_raw}, end={end_row}")
        if room_id not in self.schedules:
            raise RoomNotFoundError(f"Комната {room_id} не найдена")

        if start_hour >= end_hour:
            raise TimeRangeError(f"не правильный временной интервал: start{start_hour} >= end{end_hour}")
            return Booking(start_hour=start_hour, end_hour=end_hour,
                       booking_id=booking_id, room_id=room_id, owner=owner)

        # TODO: split по '|'
        # TODO: ожидать 5 частей: booking_id, room_id, owner, start_raw, end_raw
        # TODO: start_raw и end_raw преобразовать в int
        # TODO: если room_id не существует -> RoomNotFoundError
        # TODO: если start_hour >= end_hour -> TimeRangeError
        # TODO: вернуть объект Booking(...)
       

    def submit(self, row):
        try:
            booking = self.parse_booking(row)
            self.schedules[booking.room_id].add_booking(booking)
            self.accepted.append(booking)
        except BookingError as e:
            self.errors.append(row, type(e).__name__, str(e))

        # TODO: внутри try вызвать parse_booking(row)
        # TODO: затем schedules[booking.room_id].add_booking(booking)
        # TODO: успех добавить в self.accepted
        # TODO: BookingError сохранить в self.errors как (row, error_type, message)
        

    def load(self, rows):
        for row in rows:
            self.submit(row)
        # TODO: вызвать submit(row) для каждой строки
        

    def busiest_room(self):
        max_hours = -1
        max_room = None
        for room_id, schedule in self.schedules.items():
            total = schedule.total_reserved_hours()
            if total > max_hours:
                max_hours = total
                max_room = room_id
        return max_room, max_hours
        # TODO: найти комнату с максимумом total_reserved_hours()
        # TODO: вернуть tuple(room_id, total_hours)

    def find_booking(self, booking_id) -> Optional[Booking]:
        for schedule in self.schedule.values():
            for booking in schedule.bookings:
                if booking.booking_id == booking_id:
                    return booking
        return None
        # TODO: вернуть Optional[Booking]
        # TODO: пройтись по всем schedules и по всем bookings внутри них
        # TODO: если booking.booking_id совпал -> вернуть booking
        # TODO: если не найдено -> вернуть None
        


service = BookingService(rooms)
service.load(rows)

print("Принятые бронирования")
for b in service.accepted:
    print(f"")

# TODO: загрузить rows через service.load(rows)
# TODO: вывести принятые бронирования
# TODO: вывести ошибки
# TODO: вывести расписание по всем комнатам
# TODO: вывести busiest_room()
# TODO: вывести find_booking('BK-102')

AttributeError: 'NoneType' object has no attribute 'room_id'